# Session 3 — Density Matrices

**Author:** PPSP lab  **·**  **Date:** 2026-05-27  **·**  **Project:** Quantum Optimal Transport course  **·**  **Status:** 🔬 Teaching

## Purpose

The **density matrix** $\rho$ is *the* central object of this course. It describes both
pure states and **mixtures**, and — crucially — its **off-diagonal** entries hold the
quantum *coherence* that classical probability cannot see. We meet purity, von Neumann
entropy, fidelity, the Bloch ball, and reconstruct a genuinely **mixed** state from noisy
hardware.

## 0. What you'll be able to do

- Build a density matrix $\rho$ and read its diagonal (probabilities) vs off-diagonal (coherence).
- Tell **pure** from **mixed** with purity $\mathrm{tr}\,\rho^2$ and von Neumann entropy.
- See why $|+\rangle$ and the maximally mixed state are *different* though a Z-measurement cannot tell them apart.
- Compare states with **fidelity** and **trace distance**.
- Reconstruct a mixed $\rho$ from noisy measurements (tomography).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.quantum import density
from qot_course.quantum.states import KET_0, KET_1, KET_PLUS

np.random.seed(42)
viz.use_course_style()

## 1. From a state to a density matrix

A pure state $|\psi\rangle$ becomes the **density matrix** $\rho = |\psi\rangle\langle\psi|$
— an operator (a matrix) rather than a vector. Why bother? Because $\rho$ can also describe
*mixtures* (classical uncertainty about which state we have), which no single vector can.

In [ ]:
rho_plus = density.density_matrix(KET_PLUS)
print("rho for |+> =\n", np.round(rho_plus, 3))
print("valid density matrix:", density.is_density_matrix(rho_plus))
viz.plot_density_matrix(rho_plus, title="rho for |+>")
plt.show()

**Read the figure.** The **diagonal** (0.5, 0.5) is exactly the Z-measurement
probabilities from S2. The **off-diagonal** 0.5 entries are the *coherence* — the
fingerprint of a genuine superposition. Keep your eye on them.

## 2. Pure vs mixed — the key lesson

Imagine someone flips a fair coin and hands you $|0\rangle$ or $|1\rangle$ accordingly.
You are *classically uncertain*. That is a **mixed state**: $\rho = \tfrac12|0\rangle\langle0|
+ \tfrac12|1\rangle\langle1| = I/2$, the **maximally mixed** state.

In [ ]:
mixed = density.maximally_mixed(2)
print("rho for I/2 =\n", np.round(mixed, 3))
viz.plot_density_matrix(mixed, title="rho for I/2 (maximally mixed)")
plt.show()

**The lesson — read both figures together.** $|+\rangle$ and $I/2$ have the **same
diagonal** (0.5, 0.5): a Z-measurement gives 50/50 for *both* — it cannot tell them apart.
But $|+\rangle$ has **off-diagonal coherences** and $I/2$ does not. That difference is
purely quantum, invisible to a single classical measurement. **This gap is the seed of the
whole course**: classical optimal transport sees only the diagonal; quantum optimal
transport sees the whole operator.

In [ ]:
for name, rho in [("|+>", rho_plus), ("I/2", mixed)]:
    print(f"{name:4s}  diagonal={np.round(np.real(np.diag(rho)),3)}  "
          f"purity={density.purity(rho):.3f}  "
          f"entropy={density.von_neumann_entropy(rho):.3f} bits")

**Read the output.** Same diagonal, but **purity** 1.0 vs 0.5 and **entropy** 0 vs 1 bit:
$|+\rangle$ is pure and certain; $I/2$ is maximally mixed and maximally uncertain.

## 3. The Bloch ball

Pure states live on the **surface** of the Bloch sphere; mixed states live **inside**;
the maximally mixed $I/2$ sits at the very **centre**. The length of the Bloch vector is a
direct readout of purity.

In [ ]:
from qiskit.visualization import plot_bloch_vector

half_mixed = 0.5 * rho_plus + 0.5 * mixed   # halfway to the centre

plot_bloch_vector(list(density.bloch_vector(rho_plus)), title="pure |+> (on the surface)")
plot_bloch_vector(list(density.bloch_vector(half_mixed)), title="partly mixed (inside)")
plot_bloch_vector(list(density.bloch_vector(mixed)), title="I/2 (at the centre)")
plt.show()

**Read the figures.** As we mix in uncertainty, the Bloch vector **shrinks inward** —
from the surface (pure), through the interior (partly mixed), to the centre ($I/2$). "How
deep inside the ball" *is* "how mixed".

## 4. Telling states apart: fidelity & trace distance

Two ways to quantify how different two states are: **fidelity** $F$ (1 = identical,
0 = perfectly distinguishable) and **trace distance** $T$ (0 = identical, 1 = perfectly
distinguishable).

In [ ]:
states = {"|0>": density.density_matrix(KET_0), "|+>": rho_plus, "I/2": mixed}
names = list(states)
print(f"{'pair':12s}{'fidelity':>10s}{'trace dist':>12s}")
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a, b = names[i], names[j]
        f = density.fidelity(states[a], states[b])
        t = density.trace_distance(states[a], states[b])
        print(f"{a+' vs '+b:12s}{f:10.3f}{t:12.3f}")

**Read the table.** $|0\rangle$ vs $|+\rangle$ are partly distinguishable; $|0\rangle$
vs $I/2$ and $|+\rangle$ vs $I/2$ sit in between. Fidelity and trace distance move in
opposite directions, as they should.

## 5. Tomography: a real (noisy) state is *mixed*

Let's *measure* a density matrix. We prepare $|+\rangle$, estimate the three Pauli
expectations $\langle X\rangle, \langle Y\rangle, \langle Z\rangle$ on a **noisy** backend
(a real device's noise model), and reconstruct
$\rho = \tfrac12(I + \langle X\rangle X + \langle Y\rangle Y + \langle Z\rangle Z)$.

In [ ]:
from qiskit import QuantumCircuit, transpile
from qot_course.hardware.runtime import get_noisy_backend

backend = get_noisy_backend()   # AerSimulator carrying a real device's noise
SHOTS = 4096


def pauli_expectation_of_plus(basis):
    qc = QuantumCircuit(1, 1)
    qc.h(0)                                  # prepare |+>
    if basis == "X":
        qc.h(0)                              # rotate X-basis -> Z
    elif basis == "Y":
        qc.sdg(0); qc.h(0)                   # rotate Y-basis -> Z
    qc.measure(0, 0)
    counts = backend.run(transpile(qc, backend), shots=SHOTS, seed_simulator=42).result().get_counts()
    n0, n1 = counts.get("0", 0), counts.get("1", 0)
    return (n0 - n1) / (n0 + n1)


r = np.array([pauli_expectation_of_plus(b) for b in ("X", "Y", "Z")])
rho_hat = density.density_from_bloch(r)
print("estimated Bloch vector:", np.round(r, 3), " | |r| =", round(float(np.linalg.norm(r)), 3))
print("purity:", round(density.purity(rho_hat), 3), " (< 1  =>  the reconstructed state is MIXED)")
viz.plot_density_matrix(rho_hat, title="Tomography of a noisy |+>")
plt.show()

**Read the figure.** The ideal $|+\rangle$ should have Bloch vector $(1,0,0)$ and purity
$1$. The noisy device returns $|r| < 1$ and **purity below 1** — the prepared state is
*mixed*. This is the honest, hands-on reason the density matrix exists: real states are
never perfectly pure, and only $\rho$ can describe them.

## 6. Dictionary update

| Classical | Quantum |
|-----------|---------|
| probability vector `p` | density matrix $\rho$ (diagonal $\Rightarrow$ classical) |
| marginal | partial trace |
| event probability $p(x)$ | Born rule $|\langle x|\psi\rangle|^2$ |
| Shannon entropy $H(p)$ | **von Neumann entropy** $S(\rho) = -\mathrm{tr}(\rho\log\rho)$ |

## 7. Your turn

1. **Interpolate.** Build $\rho(t) = (1-t)\,|+\rangle\langle+| + t\,I/2$ for $t \in [0,1]$;
   plot purity and entropy vs $t$. Where does the Bloch vector go?
2. **Coherence is fragile.** Compare `plot_density_matrix` of $|+\rangle$ and of your noisy
   `rho_hat`: which entries shrank most — the diagonal or the off-diagonal?
3. **Distinguishing.** Compute the trace distance between $|+\rangle$ and `rho_hat`. How far
   did the noise push the state?

## Conclusions & references

- $\rho$ describes pure *and* mixed states; **diagonal = probabilities, off-diagonal = coherence**.
- **Purity** and **von Neumann entropy** measure mixedness; **fidelity**/**trace distance** compare states.
- Real hardware yields **mixed** states (purity < 1) — why $\rho$ is indispensable.
- **Next (S4 — Composite systems & channels):** two qubits, partial trace, entanglement, and noise as a quantum channel.

**References:** Nielsen & Chuang, *Quantum Computation and Quantum Information*, ch. 2 & 8; M. Wilde, *Quantum Information Theory*, ch. 4–5. Previous: `notebooks/01_QuantumFoundations/01_qubits.ipynb`.